<h4>Init part of importing</h4>
<p>Two parts:</p>
<ul>
    <li><b>Connection to spark:</b>  The first piece of code.</li>
    <li><b>Importing and parsing of dataframes</b> The second piece of code.</li>
</ul>

In [1]:
# Import the basic spark library
from pyspark.sql import SparkSession

# Create an entry point to the PySpark Application
spark = SparkSession.builder \
      .master("local") \
      .appName("ProjSMBUD") \
      .getOrCreate()
# master contains the URL of your remote spark instance or 'local'

Picked up _JAVA_OPTIONS: -Dawt.useSystemAAFontSettings=on -Dswing.aatext=true
Picked up _JAVA_OPTIONS: -Dawt.useSystemAAFontSettings=on -Dswing.aatext=true
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


22/12/12 20:51:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Load DataFrames (we prefer directly them to the rdd, because we already had csv with headers)
#Configuring them with some examples to check the correctness (now commented before and after the casting)

df_incollection = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ";")\
                .csv("incollectionFormatted.csv.txt")
df_book = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ";")\
                .csv("bookFormatted.csv.txt")
df_author = spark.read.option("header", True).option("encoding", "ISO-8859-1").option("delimiter", ",")\
                .csv("authorFormatted.csv.txt")

#Load bridge tables? (No, now they are in the entities), #they had already id string types, but now we don't have them





# We use of datatypes from pyspark.sql.types
from pyspark.sql.types import StringType, DateType, FloatType, StructType, StructField, ArrayType
from pyspark.sql.functions import * 
#import  org.apache.spark.sql.catalyst.expressions.objects.AssertNotNull



##We use the ids in string format!
##TO HAVE THE ARRAY OF STRINGS
df_book = df_book\
    .select("id","date", "url", "isbn", "citationscore", "publisher" , "title", split(col("editor_array"),",").alias("editor_array"))
df_incollection = df_incollection\
    .select("id","title","date", "url", "pages", "citationscore", \
    "book_id", split(col("author_array"),",").alias("author_array"), \
    split(col("cited_inc_array"),",").alias("cited_inc_array"))

df_incollection.printSchema()
#df_incollection.limit(5).select("id", 'author_array').show()
df_incollection = df_incollection \
  .withColumn("id" ,
              df_incollection["id"]
              .cast(StringType()))   \
  .withColumn("title"  ,
              df_incollection["title"]
              .cast(StringType())) \
  .withColumn("date",
              df_incollection["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_incollection["url"]
              .cast(StringType())) \
  .withColumn("pages"  ,
              df_incollection["pages"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_incollection["citationscore"]
              .cast(FloatType())) \

df_incollection.printSchema()
#df_incollection.filter(array_contains(df_incollection.author_array,"7370")).select("id","author_array").show(truncate = False)
#5371|[7371,  7370,  7372]
#df_incollection.select(explode(df_incollection.cited_inc_array)).show()



#df_book.printSchema()
#df_book.show()
df_book = df_book \
  .withColumn("id" ,
              df_book["id"]
              .cast(StringType()))   \
  .withColumn("isbn"  ,
              df_book["isbn"]
              .cast(StringType())) \
  .withColumn("date",
              df_book["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_book["url"]
              .cast(StringType())) \
  .withColumn("publisher"  ,
              df_book["publisher"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_book["citationscore"]
              .cast(FloatType())) \
  .withColumn("title"  ,
              df_book["title"]
              .cast(StringType())) \
#df_book.printSchema()
#df_book.select(explode(df_book.editor_array)).show()


#df_author.printSchema()
#df_author.limit(5).show()
df_author = df_author \
  .withColumn("id" ,
              df_author["id"]
              .cast(StringType()))   \
  .withColumn("name"  ,
              df_author["name"]
              .cast(StringType())) \
  .withColumn("citationscore",
              df_author["citationscore"]
              .cast(FloatType()))    \
#email is already a string
#some tables with a lot of columns cannot be showed well because of jupyer lab
#the special character are supported here
#df_author.printSchema() 
#df_author.limit(5).show()



root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date: string (nullable = true)
 |-- url: string (nullable = true)
 |-- pages: string (nullable = true)
 |-- citationscore: string (nullable = true)
 |-- book_id: string (nullable = true)
 |-- author_array: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- cited_inc_array: array (nullable = true)
 |    |-- element: string (containsNull = false)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date: date (nullable = true)
 |-- url: string (nullable = true)
 |-- pages: string (nullable = true)
 |-- citationscore: float (nullable = true)
 |-- book_id: string (nullable = true)
 |-- author_array: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- cited_inc_array: array (nullable = true)
 |    |-- element: string (containsNull = false)



<h4>Commands:</h4>
<ul>
    <li><b>Command 1:</b> Delete</li>
    <li><b>Command 2:</b> Creation1</li>
    <li><b>Command 3:</b> Creation2</li>
    <li><b>Command 4:</b> Update1</li>
    <li><b>Command 5:</b> Update2</li>
</ul>

In [3]:
#Delete1 OK - FRA (plus ANOTHER delete) 
#To show the difference between the deleted table instead of the original table I print both of them
#Delete all the rows of the df_incollection that has a value that is null.
df_incollection.select("id", "author_array").show(5, False)
df_incollection = df_incollection.na.drop("any")
df_incollection.select("id", "author_array").show(5, False)
#Delete all the rows of the df_book that has a value that is null.
df_book.select("id", "editor_array").show(5, False)
df_book = df_book.na.drop("any")
df_book.select("id", "editor_array").show(5, False)



+----+------------------+
|id  |author_array      |
+----+------------------+
|5367|[7367]            |
|5368|null              |
|5369|[7368]            |
|5370|[7369]            |
|5371|[7371, 7370, 7372]|
+----+------------------+
only showing top 5 rows

+----+------------------+
|id  |author_array      |
+----+------------------+
|5367|[7367]            |
|5369|[7368]            |
|5370|[7369]            |
|5371|[7371, 7370, 7372]|
|5372|[7368]            |
+----+------------------+
only showing top 5 rows

+----+------------------------------------------------+
|id  |editor_array                                    |
+----+------------------------------------------------+
|5067|null                                            |
|5068|null                                            |
|5069|[5180]                                          |
|5070|[5052, 5180]                                    |
|5071|[5182, 5187, 5186, 5185, 5181, 5184, 5183, 5188]|
+----+----------------------------

<h4>Queries:</h4>
<ul>
    <li><b>Query 1:</b> WHERE, JOIN </li>
    <li><b>Query 2:</b> WHERE, LIMIT, LIKE</li>
    <li><b>Query 3:</b> WHERE, IN, Nested Query</li>
    <li><b>Query 4:</b> GROUP BY, 1 JOIN, AS</li>
    <li><b>Query 5:</b> WHERE, GROUP BY </li>
    <li><b>Query 6:</b> GROUP BY, HAVING, AS</li>
    <li><b>Query 7:</b> WHERE, GROUP BY, HAVING, AS</li>
    <li><b>Query 8:</b> WHERE, Nested Query (i.e., 2-step Queries), GROUP BY </li>
    <li><b>Query 9:</b> WHERE, GROUP BY, HAVING, 1 JOIN</li>
    <li><b>Query 10:</b> WHERE, GROUP BY, HAVING, 2 JOINs</li>

</ul>

In [4]:
#<h4>Filtering operations (i.e., WHERE conditions)</h4> dal prof
#WHERE OK, LIMIT OK, LIKE OK -FRA
#Finding all the author with the name: Paul, ordering them on the citationscore and limiting the list to the top five
df_author.filter(col("name").like("Paul%")).orderBy("citationscore").limit(5).show(truncate = False)

#Unlike sort(), the orderBy() function guarantees a total order in the output. 
#This happens because the data will be collected into a single executor in order to be sorted.
#This means that orderBy() is more inefficient compared to sort(), but more precise, so since the query is light we use orderBy.

+----+----------------+-------------+-------------------------+
|id  |name            |citationscore|email                    |
+----+----------------+-------------+-------------------------+
|7538|Paul Leyland    |0.63710874   |paul.leyland@gmail.com   |
|2359|Paul B. Hermanns|0.7432279    |paul.b.hermanns@gmail.com|
|2337|Paul Ferring    |0.8216445    |paul.ferring@gmail.com   |
|3168|Paul Firenze    |0.8230778    |paul.firenze@gmail.com   |
|7515|Paul England    |0.84346664   |paul.england@gmail.com   |
+----+----------------+-------------+-------------------------+



In [5]:
#WHERE OK, GROUP BY OK, HAVING (is the condition after the group), AS (is the column renamed) OK - FRA

#This query first selects all the books with date > of 2000-12-12,
#then groups the values: title, date and editor_array (to then show them),
#thanks to this group creates another column showing the number of publishers
#to filter rows with number of publishers greater than 2
#and then show title, date, publishers and number of publishers.
df_book.filter(col("date")>lit("2000-02-03")\
               .cast(DateType())).groupby("title","date","editor_array")\
               .agg(size("editor_array").alias("num_of_editors"))\
               .where(col("num_of_editors") > 2 )\
               .withColumnRenamed("editor_array","editors")\
               .show(truncate = True)

#The entire query in a row:
#df_book.filter(col("date")>lit("1900-02-03").cast(DateType())).groupby("title","date","editor_array").agg(size("editor_array").alias("num_of_editors")).where(col("num_of_editors") > 2 ).withColumnRenamed("editor_array","editors").show(truncate = True)

+--------------------+----------+--------------------+--------------+
|               title|      date|             editors|num_of_editors|
+--------------------+----------+--------------------+--------------+
|Handbook of Visua...|2019-10-19|  [5289, 5288, 5290]|             3|
|Handbook of Model...|2022-01-03|[5211, 5210, 5213...|             4|
|Handbook of Ambie...|2012-03-28|[5182, 5187, 5186...|             8|
|Handbook of Real-...|2017-07-05|  [5227, 5226, 5225]|             3|
|Handbook of Ellip...|2017-09-25|[5258, 5257, 5262...|             7|
|Encyclopedia of GIS.|2020-04-03|  [5283, 5284, 5282]|             3|
|Handbook of Dynam...|2020-12-29|  [5334, 5335, 5333]|             3|
|Handbuch Industri...|2022-03-16|  [5342, 5343, 5341]|             3|
|Handbook of Heuri...|2018-11-05|  [5328, 5327, 5313]|             3|
|Handbook of Knowl...|2021-07-31|  [5305, 5307, 5306]|             3|
|Handbook of Compu...|2017-05-16|[5322, 5320, 5318...|             5|
|Handbuch Industri..